In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde # Import for KDE

# Define the lists for d and t values
d_values = [3, 4, 5]
t_values = [1, 2]

# Define the OOC data types and their corresponding file names
ooc_types = {
    'hetero': {
        'filename': 'OOC_H_values_Spline_Simulations_New_1_d{d}_t{t}.csv',
        'label': 'Hetero'
    },
    'fBm': {
        'filename': 'OOC_H_values_Spline_Simulations_New_2_d{d}_t{t}.csv',
        'label': 'fBm'
    },
    'gn': {
        'filename': 'OOC_H_values_Spline_Simulations_New_3_d{d}_t{t}.csv',
        'label': 'GN'
    }
}

def plot_histogram(ax, ic_data, ooc_data, d, t, ooc_type_label):
    """
    Plots the histograms with KDE curves for a given IC and OOC dataset on a single subplot.
    
    Args:
        ax (matplotlib.axes.Axes): The subplot axis to draw on.
        ic_data (np.ndarray): The H_value data for the In-Control process.
        ooc_data (np.ndarray): The H_value data for the Out-of-Control process.
        d (int): The d-value for the dataset.
        t (int): The t-value for the dataset.
        ooc_type_label (str): The label for the OOC data type (e.g., 'Hetero').
    """
    # Determine the bins for the histograms using all data 
    all_data = np.concatenate([ic_data, ooc_data])
    bins = np.linspace(all_data.min(), all_data.max(), 50) 
    
    # Plot the histograms 
    ax.hist(ic_data, bins=bins, density=True, alpha=0.4, label='IC (Histogram)', color='blue', edgecolor='black')
    ax.hist(ooc_data, bins=bins, density=True, alpha=0.4, label='OOC (Histogram)', color='red', edgecolor='black')
    
    # Plot KDE curves
    x_kde = np.linspace(all_data.min(), all_data.max(), 500) 
    
    kde_ic = gaussian_kde(ic_data)
    ax.plot(x_kde, kde_ic(x_kde), color='blue', linewidth=2, label='IC (KDE)')
    
    kde_ooc = gaussian_kde(ooc_data)
    ax.plot(x_kde, kde_ooc(x_kde), color='red', linestyle='--', linewidth=2, label='OOC (KDE)')
    
    ax.set_title(fr'$d={d}, \tau={t}$', fontsize=14)
    ax.set_xlabel('$H$', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.legend(loc='upper left', fontsize=20) 
    ax.grid(False)


# --- Plotting IC distribution plots ---
def plot_ic_histogram(ax, ic_data, d, t):
    bins = np.linspace(ic_data.min(), ic_data.max(), 50)
    x_kde = np.linspace(ic_data.min(), ic_data.max(), 500)
    
    ax.hist(ic_data, bins=bins, density=True, alpha=0.4, label='IC (Histogram)', color='blue', edgecolor='black')
    kde_ic = gaussian_kde(ic_data)
    ax.plot(x_kde, kde_ic(x_kde), color='blue', linewidth=2, label='IC (KDE)')
    
    ax.set_title(fr'$d={d}, \tau={t}$', fontsize=14)
    ax.set_xlabel('$H$', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.legend(loc='upper left', fontsize=20)
    ax.grid(False)

for d in d_values:
    print(f"\nProcessing Histogram plots for IC ({d})...")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
    
    try:
        ic_filename_t1 = f'IC_H_values_Spline_Simulations_d{d}_t1.csv'
        data_ic_t1 = pd.read_csv(ic_filename_t1)
        plot_ic_histogram(axes[0], data_ic_t1['H_value'], d, 1)
        
        ic_filename_t2 = f'IC_H_values_Spline_Simulations_d{d}_t2.csv'
        data_ic_t2 = pd.read_csv(ic_filename_t2)
        plot_ic_histogram(axes[1], data_ic_t2['H_value'], d, 2)
        
        plt.tight_layout()
        plt.savefig(f'Histogram_IC_d{d}.png')
        plt.close(fig)
        
    except FileNotFoundError as e:
        print(f"  Error: File not found - {e}. Skipping plots for d={d}.")
        plt.close(fig)


# --- Plotting IC vs OOC distribution plots ---
for ooc_name, ooc_info in ooc_types.items():
    print(f"\nProcessing Histogram plots with KDE for IC vs. OOC ({ooc_info['label']})...")
    
    for d in d_values:
        print(f"  Creating plot for d={d}...")
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
        
        try:
            ic_filename_t1 = f'IC_H_values_Spline_Simulations_d{d}_t1.csv'
            ooc_filename_t1 = ooc_info['filename'].format(d=d, t=1)
            
            data_ic_t1 = pd.read_csv(ic_filename_t1)
            data_ooc_t1 = pd.read_csv(ooc_filename_t1)
            
            plot_histogram(axes[0], data_ic_t1['H_value'], data_ooc_t1['H_value'], d, 1, ooc_info['label'])
            
            ic_filename_t2 = f'IC_H_values_Spline_Simulations_d{d}_t2.csv'
            ooc_filename_t2 = ooc_info['filename'].format(d=d, t=2)
            
            data_ic_t2 = pd.read_csv(ic_filename_t2)
            data_ooc_t2 = pd.read_csv(ooc_filename_t2)
            
            plot_histogram(axes[1], data_ic_t2['H_value'], data_ooc_t2['H_value'], d, 2, ooc_info['label'])
            
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.savefig(f'Histogram_Comparison_d{d}_{ooc_name}.png')
            plt.close()

        except FileNotFoundError as e:
            print(f"  Error: File not found - {e}. Skipping plots for d={d}.")
            plt.close(fig)

print("\nAll Histogram plots with KDE generated and saved successfully.")